# 31. 화면·검색·원문 보완 결과 확인

아래 셀은 저장된 결과만 읽습니다. 모델 실행·API 호출·DB 조회·파일 저장은 없습니다. `sequence_completion_guide.md`의 설명과 함께 확인하세요. 수정 전 평가와 수정 후 회귀 확인은 다른 결과입니다.


In [ ]:
# [프로젝트 추가] 실행 폴더와 관계없이 이 프로젝트의 저장 결과를 찾습니다.
from pathlib import Path
import json
프로젝트_폴더 = next((p for p in [Path.cwd(), *Path.cwd().parents]
    if (p / 'src/car_search_rag/zzong_santafe_lag/sequence_completion_guide.md').exists()), None)
if 프로젝트_폴더 is None:
    raise FileNotFoundError('이 프로젝트 안에서 노트북을 열어 주세요.')
개인_폴더 = 프로젝트_폴더 / 'src/car_search_rag/zzong_santafe_lag'
기존_결과 = json.loads((개인_폴더 / 'sequence_summary_20261005.json').read_text(encoding='utf-8'))
print('실제 화면 조건:', 기존_결과['ui']['passed'], '/', 기존_결과['ui']['total'])
print('어제 생성 사용 토큰:', 기존_결과['generation_tokens'])


In [ ]:
# [프로젝트 추가] Precision의 경계 사례와 사람 교차 판정 전 상태를 함께 보여줍니다.
지표 = 기존_결과['search']['metrics']['openai']
print('수정 전 기대 ID Hit@5:', 지표['expected_id_hit_at_5'])
print('수정 전 내용 MRR@5:', 지표['content_mrr_at_5'])
print('잠정 Precision@5 범위:', 지표['precision_at_5_lower'], '~', 지표['precision_at_5_upper'])
print('사람 교차 확인:', 기존_결과['search']['human_confirmed'])
for 문제 in 기존_결과['search']['cases']:
    if 문제['backend'] == 'openai' and not 문제['selected_has_direct_support']:
        print('발견한 문제:', 문제['case_id'], 문제['question'])


In [ ]:
# [프로젝트 추가] 저장된 최종 결과의 핵심만 보여줍니다. 전체 상세는 JSON 파일에서 확인하세요.
수정_결과_파일 = 개인_폴더 / 'review_revision_summary_20261006.json'
if 수정_결과_파일.exists():
    수정_결과 = json.loads(수정_결과_파일.read_text(encoding='utf-8'))
    저장 = 수정_결과['storage']
    print('DB 저장 완료:', 저장['db_written'])
    print('수정 주제 / 청크 / 그림 연결:', 저장['parents'], '/', 저장['chunks'], '/', 저장['image_links'])
    print('기존 개인 테이블 보존:', 저장['legacy_tables_unchanged'])
    print('회귀 확인:', 수정_결과['preflight']['v2_passed'], '/', 수정_결과['preflight']['v2_total'])
    print('저장 후 검색:', 수정_결과['db_readback']['passed'], '/', 수정_결과['db_readback']['total'])
    print('최종 화면 답변:', 수정_결과['ui_and_prompt_compare']['final_browser_status'])
    print('개인 DB / 그림 포함 합계(MB):', round(수정_결과['sizes']['personal_db_total_bytes'] / 1_000_000, 2), '/', round(수정_결과['sizes']['measured_personal_combined_bytes'] / 1_000_000, 2))
    print('검토 / 남은 초안:', 수정_결과['effective_source']['visually_reviewed_parents'], '/', 수정_결과['effective_source']['auto_draft_needs_review'])
    print('주의: 회귀 확인은 전체 PDF의 정확도나 독립 평가가 아닙니다.')
else:
    print('최종 결과 파일 작성 전입니다. 기존 결과를 새 실행 결과로 표시하지 않습니다.')


## 코드리뷰에서 설명할 때

작은 청크로 검색한 뒤 부모 전체와 필요한 주의사항을 읽습니다. 그림 파일 업로드와 그림 설명 검토는 다릅니다. 검토 버전은 수정한 부모·청크·벡터만 따로 저장하고 같은 모델의 벡터끼리 비교합니다. 이 노트북 자체는 새 실험을 실행하지 않습니다.
